In [ ]:
# 프로젝트 경로 설정
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta

# 경로 설정
table_dir = str(PROCESSED_DIR / 'AEGIS-Order-Test/de')

# 환자별 노이즈 정보
patient_mapping_df = pd.read_csv(str(PATIENT_MAPPING_PATH), sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
patient_mapping_df['new_pt_id'] = patient_mapping_df['new_pt_id'].astype(str)

patient_id_map = patient_mapping_df.set_index('pt_id')['new_pt_id'].to_dict()
date_offset_map = patient_mapping_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_offset_map = patient_mapping_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(table_dir, '**/*.csv'), recursive = True)
print(f"[INFO] 총 처리 대상 파일 수 : {len(csv_files)}")


In [ ]:
# 컬럼 삭제
print("\n[INFO] 컬럼 제거")

columns_to_drop = [
    'test_result_text', 'test_dig', 'gross'
]

for file in csv_files:
    try:
        test_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        total_cols_to_remove = len(columns_to_drop)
        present_columns = [col for col in columns_to_drop if col in test_orders_df.columns]
        removed_cols_count = len(present_columns)

        if removed_cols_count > 0:
            test_orders_df.drop(columns = present_columns, inplace = True)

        test_orders_df.to_csv(file, index=False, encoding = 'utf-8')

        print(f"[INFO] {file} - 제거 대상: {total_cols_to_remove}개, 실제 제거: {removed_cols_count}개")

    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
# 1. pt_id 치환
print("\n[INFO] 1. pt_id 치환 시작")
for file in csv_files:
    try:
        test_orders_df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')

        if 'pt_id' in test_orders_df.columns:
            test_orders_df['pt_id'] = test_orders_df['pt_id'].map(patient_id_map).fillna(test_orders_df['pt_id'])

        print(test_orders_df.head(3))

        test_orders_df.to_csv(file, index=False, encoding = 'utf-8')
        print(f"[INFO] pt_id 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] pt_id 오류: {file} → {e}")


In [ ]:
# 2. hos_dt ±7일 ±30분
print("\n[INFO] 2. hos_dt 노이즈 적용")
for file in csv_files:
    try:
        test_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'hos_dt' in test_orders_df.columns:
            test_orders_df['hos_dt'] = pd.to_datetime(test_orders_df['hos_dt'], errors='coerce')
            test_orders_df['hos_dt'] = test_orders_df['hos_dt'] + test_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(test_orders_df.head(3))

            test_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] hos_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
# order_date ±7일 ±30분
print("\n[INFO] order_date 노이즈 적용")
for file in csv_files:
    try:
        test_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'order_date' in test_orders_df.columns:
            test_orders_df['order_date'] = pd.to_datetime(test_orders_df['order_date'], errors='coerce')
            test_orders_df['order_date'] = test_orders_df['order_date'] + test_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(test_orders_df.head(3))

            test_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] order_date 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] order_date 오류: {file} → {e}")


In [ ]:
# register_dt ±7일 ±30분
print("\n[INFO] register_dt 노이즈 적용")
for file in csv_files:
    try:
        test_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'register_dt' in test_orders_df.columns:
            test_orders_df['register_dt'] = pd.to_datetime(test_orders_df['register_dt'], errors='coerce')
            test_orders_df['register_dt'] = test_orders_df['register_dt'] + test_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(test_orders_df.head(3))

            test_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] register_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] register_dt 오류: {file} → {e}")


In [ ]:
# conduct_dt ±7일 ±30분
print("\n[INFO] conduct_dt 노이즈 적용")
for file in csv_files:
    try:
        test_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'conduct_dt' in test_orders_df.columns:
            test_orders_df['conduct_dt'] = pd.to_datetime(test_orders_df['conduct_dt'], errors='coerce')
            test_orders_df['conduct_dt'] = test_orders_df['conduct_dt'] + test_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(test_orders_df.head(3))

            test_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] conduct_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] conduct_dt 오류: {file} → {e}")


In [ ]:
# reading_dt ±7일 ±30분
print("\n[INFO] reading_dt 노이즈 적용")
for file in csv_files:
    try:
        test_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'reading_dt' in test_orders_df.columns:
            test_orders_df['reading_dt'] = pd.to_datetime(test_orders_df['reading_dt'], errors='coerce')
            test_orders_df['reading_dt'] = test_orders_df['reading_dt'] + test_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(test_orders_df.head(3))

            test_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] reading_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] reading_dt 오류: {file} → {e}")


In [ ]:
# test_result_report_dt ±7일 ±30분
print("\n[INFO] test_result_report_dt 노이즈 적용")
for file in csv_files:
    try:
        test_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'test_result_report_dt' in test_orders_df.columns:
            test_orders_df['test_result_report_dt'] = pd.to_datetime(test_orders_df['test_result_report_dt'], errors='coerce')
            test_orders_df['test_result_report_dt'] = test_orders_df['test_result_report_dt'] + test_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(test_orders_df.head(3))

            test_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] test_result_report_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] test_result_report_dt 오류: {file} → {e}")


In [ ]:
# report_dt ±7일 ±30분
print("\n[INFO] report_dt 노이즈 적용")
for file in csv_files:
    try:
        test_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'report_dt' in test_orders_df.columns:
            test_orders_df['report_dt'] = pd.to_datetime(test_orders_df['report_dt'], errors='coerce')
            test_orders_df['report_dt'] = test_orders_df['report_dt'] + test_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(test_orders_df.head(3))

            test_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] report_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] report_dt 오류: {file} → {e}")


In [ ]:
# tx_dt ±7일 ±30분
print("\n[INFO] tx_dt 노이즈 적용")
for file in csv_files:
    try:
        test_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'tx_dt' in test_orders_df.columns:
            test_orders_df['tx_dt'] = pd.to_datetime(test_orders_df['tx_dt'], errors='coerce')
            test_orders_df['tx_dt'] = test_orders_df['tx_dt'] + test_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(test_orders_df.head(3))

            test_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] tx_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] tx_dt 오류: {file} → {e}")
